# ADC Example 7 — Dual ADC Interleaved Mode

**Target:** STM32F103C8T6 · ADC1 + ADC2 · Interleaved sampling
**Focus:** Double the effective sample rate by having ADC1 and ADC2 take turns converting the same channel.

---
## Concept

A single F103 ADC running at ADCCLK = 12 MHz has a practical ceiling of about **857 kSPS** (14 ADCCLK cycles per conversion)[reference:13]. In **interleaved dual mode**, ADC1 and ADC2 alternate conversions on the same channel, and the combined result stream arrives twice as fast — up to **~1.71 MSPS**[reference:14].

The data from both ADCs is packed into **ADC1->DR** as a **32-bit word**: the lower 16 bits contain ADC1's result, the upper 16 bits contain ADC2's result. DMA must therefore be configured for **Word (32-bit)** transfers, not Half Word.

---
## CubeMX Limitation

**STM32CubeMX for F1 does not expose `DUALMOD` in the ADC configuration pane.** You must:
1. Configure ADC1 and ADC2 independently in CubeMX.
2. Generate code.
3. **Manually write `DUALMOD[3:0]` in `ADC1->CR1`** in user code, after calibrating both ADCs.

---
## CubeMX Configuration

### ADC1
| Parameter | Setting |
| :--- | :--- |
| `Scan Conversion Mode` | Disabled |
| `Continuous Conversion Mode` | Disabled (interleaved mode manages the rhythm) |
| `External Trigger Conversion Source` | **Timer 3 Trigger Out event** |
| Rank 1 | Channel 5, Sampling Time **1.5 cycles** (minimum, for max speed) |

### ADC2
| Parameter | Setting |
| :--- | :--- |
| `Scan Conversion Mode` | Disabled |
| `Continuous Conversion Mode` | Disabled |
| Rank 1 | Same channel as ADC1 (Channel 5) |

### DMA (ADC1 only)
- Add **ADC1** → DMA1 Channel 1.
- **Mode:** `Circular`.
- **Data Width:** **Word (32-bit)** — critical for interleaved mode[reference:15].
- **Memory Increment:** Enabled.

### TIM3
- Prescaler and ARR set to produce the desired **trigger rate**. In interleaved mode, each trigger starts a pair of conversions (one on ADC1, one on ADC2).

> **Do not** enable continuous mode on either ADC. The interleaved pair is initiated by the external trigger and managed by the dual-mode hardware.

## Required User Code — Calibration and DUALMOD

### Step 1 — Calibrate both ADCs

Calibration must be performed **before** writing `DUALMOD`.

In [ ]:
HAL_ADCEx_Calibration_Start(&hadc1);
HAL_ADCEx_Calibration_Start(&hadc2);

### Step 2 — Write DUALMOD[3:0] in ADC1->CR1

`DUALMOD` occupies **bits 19:16 of `ADC1->CR1`**. For **fast interleaved mode only**, the encoding is **`0111`** (`0x7`)[reference:17]. The code below clears the field and writes the interleaved-mode value.

In [ ]:
// Clear DUALMOD[3:0] (bits 19:16) and write fast-interleaved mode (0111)
ADC1->CR1 &= ~(0xFu << 16);
ADC1->CR1 |=  (0x7u << 16);   // DUALMOD = 0111: fast interleaved, regular group only

### DUALMOD Encoding Reference (RM0008)

| DUALMOD[3:0] | Mode |
| :--- | :--- |
| `0000` | Independent mode |
| `0111` | **Fast interleaved, regular group only** |
| `0100` | Slow interleaved, regular group only |
| `0011` | Combined injected simultaneous + fast interleaved |
| `0010` | Combined injected simultaneous + slow interleaved |

> These encodings are defined in RM0008 §11.12.2 and the CMSIS device header `stm32f103xb.h`[reference:18].

### Step 3 — Start Timer and DMA

Start the trigger source first, then start ADC1 in DMA mode. The interleaved pair is generated on each TRGO pulse.

In [ ]:
#define DUAL_BUF_LEN  2048
uint32_t dual_buf[DUAL_BUF_LEN];   // 32-bit words, NOT uint16_t

HAL_TIM_Base_Start(&htim3);
HAL_ADC_Start_DMA(&hadc1, (uint32_t *)dual_buf, DUAL_BUF_LEN);

// Each 32-bit word:
//   bits 15:0   = ADC1 result
//   bits 31:16  = ADC2 result

## Unpacking the Interleaved Stream

Each `dual_buf[i]` contains one ADC1 sample and one ADC2 sample taken half a conversion cycle apart. Unpack them into separate arrays for processing.

In [ ]:
void unpack_dual(const uint32_t *src, uint16_t *adc1_out, uint16_t *adc2_out, uint32_t n) {
    for (uint32_t i = 0; i < n; i++) {
        adc1_out[i] = (uint16_t)(src[i] & 0xFFFF);        // lower half-word
        adc2_out[i] = (uint16_t)((src[i] >> 16) & 0xFFFF); // upper half-word
    }
}

// Usage:
uint16_t adc1_samples[DUAL_BUF_LEN];
uint16_t adc2_samples[DUAL_BUF_LEN];
unpack_dual(dual_buf, adc1_samples, adc2_samples, DUAL_BUF_LEN);

## Timing Budget

| Parameter | Single ADC | Dual Interleaved |
| :--- | :--- | :--- |
| ADCCLK | 12 MHz | 12 MHz |
| Cycles per conversion | 14 | 14 (per ADC, alternating) |
| Effective sample period | 14 / 12 MHz ≈ **1.17 µs** | **~0.58 µs** (half) |
| Max sample rate | **857 kSPS** | **~1.71 MSPS**[reference:19] |

## Critical Notes

- **DMA data width must be Word (32-bit).** Using Half Word in interleaved mode truncates the ADC2 result and corrupts unpacking[reference:20].
- **Calibrate both ADCs separately** before enabling dual mode.
- **DUALMOD must be written while the ADCs are disabled** or before starting conversions. If you change it at runtime, stop both ADCs first.
- The F103 reference manual (RM0008) devotes only a few pages to dual mode; the CubeMX F1 interface does not generate the `DUALMOD` write, so the register-level step above is mandatory.